In [6]:
pip install fastapi uvicorn openai sse-starlette nest_asyncio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from sse_starlette.sse import EventSourceResponse
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    description: str

items = [
    Item(name="Plumbus", description="A multi-purpose household device."),
    Item(name="Portal Gun", description="A portal opening device."),
    Item(name="Meeseeks Box", description="A box that summons a Meeseeks."),
]

# 1. Asynchronous Generator with an artificial delay
async def item_publisher():
    try:
        for item in items:
            await asyncio.sleep(2)
            yield {"data": item.model_dump_json()}
        yield {"event": "done", "data": "Stream finished"}
    except asyncio.CancelledError:
        print("Client disconnected early.")

# 2. The SSE Endpoint
@app.get("/items/stream")
async def sse_items():
    return EventSourceResponse(item_publisher())

# 3. HTML Frontend to visualize the stream in real-time
@app.get("/", response_class=HTMLResponse)
async def get_frontend():
    return """
    <!DOCTYPE html>
    <html>
    <head>
        <title>SSE Demo</title>
        <style>
            body { font-family: Arial, sans-serif; margin: 40px; background: #f4f4f9; color: #333; }
            h2 { color: #2c3e50; }
            #stream-box { border: 2px dashed #b2bec3; padding: 20px; background: white; min-height: 150px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }
            .item { padding: 12px; margin: 10px 0; background: #e2f0cb; border-left: 5px solid #88d49e; border-radius: 4px; font-size: 1.1em; animation: fadeIn 0.4s ease-out; }
            .status { color: #7f8c8d; font-style: italic; margin-top: 15px; display: block; }
            @keyframes fadeIn { from { opacity: 0; transform: translateY(8px); } to { opacity: 1; transform: translateY(0); } }
        </style>
    </head>
    <body>
        <h2>Real-Time FastAPI SSE Stream</h2>
        <p>Items will appear below one by one as they are pushed from the server (every 2 seconds):</p>
        <div id="stream-box">Waiting for stream to start...</div>
        <span id="status-text" class="status">Connecting to server...</span>

        <script>
            const eventSource = new EventSource("/items/stream");
            const box = document.getElementById("stream-box");
            const statusText = document.getElementById("status-text");
            
            let first = true;

            eventSource.onmessage = function(event) {
                if (first) { box.innerHTML = ""; first = false; }
                statusText.innerText = "Receiving stream...";
                const item = JSON.parse(event.data);
                box.innerHTML += `<div class='item'><strong>📦 ${item.name}</strong> — ${item.description}</div>`;
            };

            eventSource.addEventListener("done", function(event) {
                statusText.innerText = "Stream completed successfully.";
                eventSource.close();
            });

            eventSource.onerror = function(err) {
                if (eventSource.readyState === EventSource.CLOSED) {
                    statusText.innerText = "Connection closed.";
                } else {
                    console.log("An error occurred or server disconnected.");
                }
            };
        </script>
    </body>
    </html>
    """

# 4. JUPYTER WORKAROUND: Run Uvicorn as a background task on Jupyter's loop
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

# This attaches the server to Jupyter's active event loop without blocking it
loop = asyncio.get_event_loop()
loop.create_task(server.serve())

<Task pending name='Task-4' coro=<Server.serve() running at c:\Users\raghav.mittal\SLK\learning\myenv\lib\site-packages\uvicorn\server.py:77>>

INFO:     Started server process [7292]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:53058 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:53058 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:64193 - "GET /items/stream HTTP/1.1" 200 OK
INFO:     127.0.0.1:60219 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:60219 - "GET /items/stream HTTP/1.1" 200 OK
INFO:     127.0.0.1:63916 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:63916 - "GET /items/stream HTTP/1.1" 200 OK
INFO:     127.0.0.1:63062 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:63062 - "GET /items/stream HTTP/1.1" 200 OK
INFO:     127.0.0.1:55323 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:55323 - "GET /items/stream HTTP/1.1" 200 OK
Client disconnected early.
